In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import nibabel as nib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from scipy.spatial.distance import cdist

device = torch.device("cpu")
print("Using device:", device)


Using device: cpu


In [2]:
DATASET_PATH = r"C:\Users\User\brats2021\BraTS2021_Training_Data"  



In [3]:
def center_crop(tensor, target_shape):
    _, _, d, h, w = tensor.shape
    td, th, tw = target_shape

    d1 = (d - td) // 2
    h1 = (h - th) // 2
    w1 = (w - tw) // 2

    return tensor[:, :, d1:d1+td, h1:h1+th, w1:w1+tw]


In [4]:
class BraTS3DDataset(Dataset):
    def __init__(self, dataset_path, max_patients=5):
        self.dataset_path = dataset_path
        self.patients = sorted([
            p for p in os.listdir(dataset_path)
            if os.path.isdir(os.path.join(dataset_path, p))
        ])[:max_patients]

    def normalize(self, vol):
        mask = vol != 0
        vol[mask] = (vol[mask] - vol[mask].mean()) / (vol[mask].std() + 1e-8)
        return vol

    def remap_labels(self, seg):
        seg = seg.copy()
        seg[seg == 4] = 3
        return seg

    def __len__(self):
        return len(self.patients)

    def __getitem__(self, idx):
        pid = self.patients[idx]
        ppath = os.path.join(self.dataset_path, pid)

        t1 = nib.load(os.path.join(ppath, f"{pid}_t1.nii.gz")).get_fdata()
        t1ce = nib.load(os.path.join(ppath, f"{pid}_t1ce.nii.gz")).get_fdata()
        t2 = nib.load(os.path.join(ppath, f"{pid}_t2.nii.gz")).get_fdata()
        flair = nib.load(os.path.join(ppath, f"{pid}_flair.nii.gz")).get_fdata()
        seg = nib.load(os.path.join(ppath, f"{pid}_seg.nii.gz")).get_fdata()

        t1 = self.normalize(t1)
        t1ce = self.normalize(t1ce)
        t2 = self.normalize(t2)
        flair = self.normalize(flair)

        # Reorder from (H, W, D) → (D, H, W)
        t1 = np.transpose(t1, (2, 0, 1))
        t1ce = np.transpose(t1ce, (2, 0, 1))
        t2 = np.transpose(t2, (2, 0, 1))
        flair = np.transpose(flair, (2, 0, 1))
        seg = np.transpose(seg, (2, 0, 1))
        
        image = np.stack([t1, t1ce, t2, flair], axis=0)

        seg = self.remap_labels(seg)

        return torch.tensor(image, dtype=torch.float32), torch.tensor(seg, dtype=torch.long)


In [5]:
dataset = BraTS3DDataset(DATASET_PATH, max_patients=1)
loader = DataLoader(dataset, batch_size=1, shuffle=False)

print("Patients loaded:", len(dataset))


Patients loaded: 1


In [6]:
class UNet3D(nn.Module):
    def __init__(self, in_channels=4, num_classes=4):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv3d(in_c, out_c, 3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv3d(out_c, out_c, 3, padding=1),
                nn.ReLU(inplace=True)
            )

        self.enc1 = block(4, 16)
        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = block(16, 32)
        self.pool2 = nn.MaxPool3d(2)

        self.bottleneck = block(32, 64)

        self.up2 = nn.ConvTranspose3d(64, 32, 2, stride=2)
        self.dec2 = block(64, 32)

        self.up1 = nn.ConvTranspose3d(32, 16, 2, stride=2)
        self.dec1 = block(32, 16)

        self.out = nn.Conv3d(16, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        d2 = self.up2(b)
        e2 = center_crop(e2, d2.shape[2:])
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        e1 = center_crop(e1, d1.shape[2:])
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)


In [7]:
# Instantiate model
unet = UNet3D(in_channels=4, num_classes=4)
unet = unet.to(device)
unet.eval()

# Take one batch from loader
image, target = next(iter(loader))

print("Input batch shape:", image.shape)
print("Target batch shape:", target.shape)

# Forward pass
with torch.no_grad():
    output = unet(image)

print("Output shape:", output.shape)


Input batch shape: torch.Size([1, 4, 155, 240, 240])
Target batch shape: torch.Size([1, 155, 240, 240])
Output shape: torch.Size([1, 4, 152, 240, 240])


In [9]:
# Crop target to match output spatial size
target_cropped = center_crop(
    target.unsqueeze(1),  # add channel dim
    output.shape[2:]      # (D, H, W)
).squeeze(1)

print("Cropped target shape:", target_cropped.shape)

# Convert model output to predicted labels
pred = torch.argmax(output, dim=1)

print("Prediction shape:", pred.shape)

# Convert to numpy for metrics
pred_np = pred.cpu().numpy()[0]
gt_np = target_cropped.cpu().numpy()[0]

print("Unique predicted labels:", np.unique(pred_np))
print("Unique GT labels:", np.unique(gt_np))


Cropped target shape: torch.Size([1, 152, 240, 240])
Prediction shape: torch.Size([1, 152, 240, 240])
Unique predicted labels: [1]
Unique GT labels: [0 1 2 3]


In [10]:
def dice_score_binary(pred, gt):
    pred = pred.astype(np.float32)
    gt = gt.astype(np.float32)

    intersection = (pred * gt).sum()
    return (2.0 * intersection) / (pred.sum() + gt.sum() + 1e-6)

# Whole Tumor = labels {1,2,3}
pred_wt = np.isin(pred_np, [1, 2, 3])
gt_wt = np.isin(gt_np, [1, 2, 3])

dice_wt = dice_score_binary(pred_wt, gt_wt)

print("Dice score (Whole Tumor):", dice_wt)


Dice score (Whole Tumor): 0.04261086271380242


In [12]:
from scipy.ndimage import binary_erosion
from scipy.spatial.distance import cdist

def extract_boundary(mask):
    """
    Extract surface voxels of a 3D binary mask
    """
    eroded = binary_erosion(mask)
    boundary = mask ^ eroded
    return np.argwhere(boundary)


def hd95_binary_safe(pred, gt):
    """
    Memory-safe HD95 computation using boundary points only
    """
    pred_pts = extract_boundary(pred)
    gt_pts = extract_boundary(gt)

    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return np.inf

    # Compute distances in chunks (extra safety)
    dists_pred = []
    for i in range(0, len(pred_pts), 5000):
        chunk = pred_pts[i:i+5000]
        dists = cdist(chunk, gt_pts)
        dists_pred.append(dists.min(axis=1))

    dists_pred = np.concatenate(dists_pred)

    dists_gt = []
    for i in range(0, len(gt_pts), 5000):
        chunk = gt_pts[i:i+5000]
        dists = cdist(chunk, pred_pts)
        dists_gt.append(dists.min(axis=1))

    dists_gt = np.concatenate(dists_gt)

    hd95_pred = np.percentile(dists_pred, 95)
    hd95_gt = np.percentile(dists_gt, 95)

    return max(hd95_pred, hd95_gt)


# Whole Tumor = labels {1,2,3}
pred_wt = np.isin(pred_np, [1, 2, 3])
gt_wt = np.isin(gt_np, [1, 2, 3])

hd95_wt = hd95_binary_safe(pred_wt, gt_wt)

print("HD95 (Whole Tumor, safe):", hd95_wt)


HD95 (Whole Tumor, safe): 141.9013742005341


In [13]:
# Tumor Core = labels {1,3}
pred_tc = np.isin(pred_np, [1, 3])
gt_tc = np.isin(gt_np, [1, 3])

dice_tc = dice_score_binary(pred_tc, gt_tc)
hd95_tc = hd95_binary_safe(pred_tc, gt_tc)

print("Dice (Tumor Core):", dice_tc)
print("HD95 (Tumor Core):", hd95_tc)


Dice (Tumor Core): 0.00794052490193706
HD95 (Tumor Core): 161.97221984031705


In [14]:
# Enhancing Tumor = label {3}
pred_et = (pred_np == 3)
gt_et = (gt_np == 3)

dice_et = dice_score_binary(pred_et, gt_et)
hd95_et = hd95_binary_safe(pred_et, gt_et)

print("Dice (Enhancing Tumor):", dice_et)
print("HD95 (Enhancing Tumor):", hd95_et)


Dice (Enhancing Tumor): 0.0
HD95 (Enhancing Tumor): inf


In [ ]:
# Get coordinates of all tumor voxels
tumor_coords = np.argwhere(seg > 0)

# Compute centroid (average location)
centroid = tumor_coords.mean(axis=0)

print("Tumor centroid (x, y, z):", centroid)

# Determine left/right hemisphere using x-axis
mid_x = seg.shape[0] / 2
hemisphere = "Left" if centroid[0] < mid_x else "Right"

print("Estimated hemisphere:", hemisphere)


Tumor centroid (x, y, z): [140.74865421  95.73413644  79.21218401]
Estimated hemisphere: Right


In [ ]:
features = {
    "tumor_present": bool(tumor_present),
    "whole_tumor_volume_mm3": float(wt_volume_mm3),
    "tumor_core_volume_mm3": float(tc_volume_mm3),
    "enhancing_tumor_volume_mm3": float(et_volume_mm3),
    "slices_with_tumor": int(slices_with_tumor),
    "total_slices": int(total_slices),
    "tumor_centroid_xyz": centroid.tolist(),
    "hemisphere": hemisphere
}

features


{'tumor_present': True,
 'whole_tumor_volume_mm3': 190594.0,
 'tumor_core_volume_mm3': 34899.0,
 'enhancing_tumor_volume_mm3': 23651.0,
 'slices_with_tumor': 86,
 'total_slices': 155,
 'tumor_centroid_xyz': [140.74865420737274,
  95.73413643661395,
  79.21218401418723],
 'hemisphere': 'Right'}

In [ ]:
def normalize_intensity(volume):
    mask = volume > 0
    mean = volume[mask].mean()
    std = volume[mask].std()
    volume_norm = (volume - mean) / (std + 1e-8)
    volume_norm[~mask] = 0
    return volume_norm

flair_norm = normalize_intensity(flair)

print("Original FLAIR stats:")
print("Mean:", flair[flair > 0].mean(), "Std:", flair[flair > 0].std())

print("\nNormalized FLAIR stats:")
print("Mean:", flair_norm[flair_norm != 0].mean(),
      "Std:", flair_norm[flair_norm != 0].std())


Original FLAIR stats:
Mean: 860.8151190616409 Std: 326.4080999789049

Normalized FLAIR stats:
Mean: -7.777050141902923e-17 Std: 0.9999999999693637


In [ ]:
import nibabel as nib
import numpy as np
import os

def load_nifti(path):
    nii = nib.load(path)
    return nii.get_fdata(), nii.header

def normalize_intensity(volume):
    mask = volume > 0
    mean = volume[mask].mean()
    std = volume[mask].std()
    volume_norm = (volume - mean) / (std + 1e-8)
    volume_norm[~mask] = 0
    return volume_norm

# Reload volumes
flair, flair_header = load_nifti(os.path.join(patient_path, f"{patient_id}_flair.nii.gz"))
t1, _    = load_nifti(os.path.join(patient_path, f"{patient_id}_t1.nii.gz"))
t1ce, _  = load_nifti(os.path.join(patient_path, f"{patient_id}_t1ce.nii.gz"))
t2, _    = load_nifti(os.path.join(patient_path, f"{patient_id}_t2.nii.gz"))
seg, seg_header = load_nifti(os.path.join(patient_path, f"{patient_id}_seg.nii.gz"))

print("All MRI volumes reloaded.")


All MRI volumes reloaded.


In [ ]:
# Normalize all MRI modalities
t1_norm = normalize_intensity(t1)
t1ce_norm = normalize_intensity(t1ce)
t2_norm = normalize_intensity(t2)
flair_norm = normalize_intensity(flair)

print("Normalization completed for all modalities.")
print("T1 mean/std:", t1_norm[t1_norm != 0].mean(), t1_norm[t1_norm != 0].std())
print("T1ce mean/std:", t1ce_norm[t1ce_norm != 0].mean(), t1ce_norm[t1ce_norm != 0].std())
print("T2 mean/std:", t2_norm[t2_norm != 0].mean(), t2_norm[t2_norm != 0].std())
print("FLAIR mean/std:", flair_norm[flair_norm != 0].mean(), flair_norm[flair_norm != 0].std())


Normalization completed for all modalities.
T1 mean/std: 1.339040654353889e-16 0.999999999931412
T1ce mean/std: -4.343717244611396e-17 0.9999999999868165
T2 mean/std: -9.797858446491872e-17 0.9999999999698818
FLAIR mean/std: -7.777050141902923e-17 0.9999999999693637


In [ ]:
# 3d cnn start here

In [ ]:
# Stack modalities along channel dimension
# Current shape: (H, W, D)
# CNN expects: (C, D, H, W)

input_volume = np.stack(
    [t1_norm, t1ce_norm, t2_norm, flair_norm],
    axis=0
)

# Rearrange axes from (C, H, W, D) -> (C, D, H, W)
input_volume = np.transpose(input_volume, (0, 3, 1, 2))

print("Input volume shape (C, D, H, W):", input_volume.shape)


Input volume shape (C, D, H, W): (4, 155, 240, 240)


In [ ]:
# Rearrange segmentation from (H, W, D) -> (D, H, W)
target_seg = np.transpose(seg, (2, 0, 1))

print("Target segmentation shape (D, H, W):", target_seg.shape)
print("Unique labels in target:", np.unique(target_seg))


Target segmentation shape (D, H, W): (155, 240, 240)
Unique labels in target: [0. 1. 2. 4.]


In [ ]:
import torch

# Convert input to tensor
# Shape before: (C, D, H, W)
# Shape after: (1, C, D, H, W)
input_tensor = torch.tensor(input_volume, dtype=torch.float32).unsqueeze(0)

# Convert target to tensor
# Shape before: (D, H, W)
# Shape after: (1, D, H, W)
target_tensor = torch.tensor(target_seg, dtype=torch.long).unsqueeze(0)

print("Input tensor shape:", input_tensor.shape)
print("Target tensor shape:", target_tensor.shape)


Input tensor shape: torch.Size([1, 4, 155, 240, 240])
Target tensor shape: torch.Size([1, 155, 240, 240])


In [ ]:
import torch.nn as nn

class Simple3DCNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(Simple3DCNN, self).__init__()

        self.conv1 = nn.Conv3d(in_channels, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool3d(kernel_size=2)

        self.conv2 = nn.Conv3d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()

        self.out_conv = nn.Conv3d(32, num_classes, kernel_size=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)

        x = self.out_conv(x)
        return x

# Instantiate model
model = Simple3DCNN(in_channels=4, num_classes=4)

print(model)


Simple3DCNN(
  (conv1): Conv3d(4, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (relu1): ReLU()
  (pool1): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (relu2): ReLU()
  (out_conv): Conv3d(32, 4, kernel_size=(1, 1, 1), stride=(1, 1, 1))
)


In [ ]:
# Run a forward pass
with torch.no_grad():
    output = model(input_tensor)

print("Model output shape:", output.shape)


Model output shape: torch.Size([1, 4, 77, 120, 120])


In [ ]:
class UNet3D(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(UNet3D, self).__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv3d(in_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv3d(out_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True)
            )

        self.enc1 = conv_block(in_channels, 16)
        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = conv_block(16, 32)
        self.pool2 = nn.MaxPool3d(2)

        self.bottleneck = conv_block(32, 64)

        self.up2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = conv_block(64, 32)

        self.up1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = conv_block(32, 16)

        self.out_conv = nn.Conv3d(16, num_classes, kernel_size=1)

    def forward(self, x):
      e1 = self.enc1(x)
      p1 = self.pool1(e1)

      e2 = self.enc2(p1)
      p2 = self.pool2(e2)

      b = self.bottleneck(p2)

      d2 = self.up2(b)
      e2_crop = center_crop(e2, d2.shape[2:])
      d2 = torch.cat([d2, e2_crop], dim=1)
      d2 = self.dec2(d2)

      d1 = self.up1(d2)
      e1_crop = center_crop(e1, d1.shape[2:])
      d1 = torch.cat([d1, e1_crop], dim=1)
      d1 = self.dec1(d1)

      out = self.out_conv(d1)
      return out

# Instantiate model
unet = UNet3D(in_channels=4, num_classes=4)
print(unet)


UNet3D(
  (enc1): Sequential(
    (0): Conv3d(4, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): ReLU(inplace=True)
    (2): Conv3d(16, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (3): ReLU(inplace=True)
  )
  (pool1): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc2): Sequential(
    (0): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): ReLU(inplace=True)
    (2): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (3): ReLU(inplace=True)
  )
  (pool2): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (bottleneck): Sequential(
    (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (1): ReLU(inplace=True)
    (2): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (3): ReLU(inplace=True)
  )
  (up2): ConvTranspose3d(64, 32, kernel_size=(2, 2, 2), stride=(2

In [ ]:
def center_crop(enc_feat, target_shape):
    _, _, d, h, w = enc_feat.shape
    td, th, tw = target_shape

    d1 = (d - td) // 2
    h1 = (h - th) // 2
    w1 = (w - tw) // 2

    return enc_feat[:, :,
                     d1:d1+td,
                     h1:h1+th,
                     w1:w1+tw]

with torch.no_grad():
    unet_output = unet(input_tensor)

print("U-Net output shape:", unet_output.shape)


U-Net output shape: torch.Size([1, 4, 152, 240, 240])


In [ ]:
from torch.utils.data import Dataset

class BraTS3DDataset(Dataset):
    def __init__(self, dataset_path, patient_ids):
        self.dataset_path = dataset_path
        self.patient_ids = patient_ids

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        patient_id = self.patient_ids[idx]
        patient_path = os.path.join(self.dataset_path, patient_id)

        # Load MRI volumes
        t1, _ = load_nifti(os.path.join(patient_path, f"{patient_id}_t1.nii.gz"))
        t1ce, _ = load_nifti(os.path.join(patient_path, f"{patient_id}_t1ce.nii.gz"))
        t2, _ = load_nifti(os.path.join(patient_path, f"{patient_id}_t2.nii.gz"))
        flair, _ = load_nifti(os.path.join(patient_path, f"{patient_id}_flair.nii.gz"))
        seg, _ = load_nifti(os.path.join(patient_path, f"{patient_id}_seg.nii.gz"))

        # Normalize modalities
        t1 = normalize_intensity(t1)
        t1ce = normalize_intensity(t1ce)
        t2 = normalize_intensity(t2)
        flair = normalize_intensity(flair)

        # Stack into (C, D, H, W)
        image = np.stack([t1, t1ce, t2, flair], axis=0)
        image = np.transpose(image, (0, 3, 1, 2))

        # Segmentation target (D, H, W)
        target = np.transpose(seg, (2, 0, 1))

        # Convert to tensors
        image = torch.tensor(image, dtype=torch.float32)
        target = torch.tensor(target, dtype=torch.long)
        target = remap_labels(target)


        return image, target


In [ ]:
from sklearn.model_selection import train_test_split

# Patient-wise split
train_ids, val_ids = train_test_split(
    patients,
    test_size=0.2,
    random_state=42
)

print("Training patients:", len(train_ids))
print("Validation patients:", len(val_ids))


Training patients: 1000
Validation patients: 250


In [ ]:
# Create Dataset objects
train_dataset = BraTS3DDataset(DATASET_PATH, train_ids)
val_dataset = BraTS3DDataset(DATASET_PATH, val_ids)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))


Train dataset size: 1000
Validation dataset size: 250


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train loader batches:", len(train_loader))
print("Validation loader batches:", len(val_loader))


Train loader batches: 1000
Validation loader batches: 250


In [ ]:
def remap_labels(seg):
    """
    BraTS labels: {0,1,2,4} → {0,1,2,3}
    """
    seg = seg.clone()
    seg[seg == 4] = 3
    return seg


In [ ]:
# Get one batch from training loader
image_batch, target_batch = next(iter(train_loader))

# Move to same device as model
image_batch = image_batch.to(device)
target_batch = target_batch.to(device)

print("Input batch shape:", image_batch.shape)
print("Target batch shape:", target_batch.shape)

# Forward pass
with torch.no_grad():
    output_batch = unet(image_batch)

print("Model output shape:", output_batch.shape)


Input batch shape: torch.Size([1, 4, 155, 240, 240])
Target batch shape: torch.Size([1, 155, 240, 240])
Model output shape: torch.Size([1, 4, 152, 240, 240])


In [ ]:
import torch.nn.functional as F

def dice_loss(pred, target, smooth=1e-5):
    """
    pred: (B, C, D, H, W) logits
    target: (B, D, H, W) labels
    """
    pred = F.softmax(pred, dim=1)

    dice = 0.7
    num_classes = pred.shape[1]

    for c in range(1, num_classes):  # ignore background
        pred_c = pred[:, c]
        target_c = (target == c).float()

        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()

        dice += 1 - (2. * intersection + smooth) / (union + smooth)

    return dice / (num_classes - 1)


In [ ]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Move model to device
unet = unet.to(device)

# Optimizer
optimizer = torch.optim.Adam(unet.parameters(), lr=1e-4)


Using device: cuda


In [ ]:
def crop_target(target, output_shape):
    """
    target: (B, D, H, W)
    output_shape: shape of model output (B, C, D_out, H, W)
    """
    _, _, d_out, h_out, w_out = output_shape
    d_in = target.shape[1]

    d1 = (d_in - d_out) // 2
    return target[:, d1:d1 + d_out, :, :]


In [ ]:
# Select subset sizes
train_ids_50 = train_ids[:50]
val_ids_10 = val_ids[:10]

# Create datasets
train_dataset_50 = BraTS3DDataset(DATASET_PATH, train_ids_50)
val_dataset_10 = BraTS3DDataset(DATASET_PATH, val_ids_10)

# Create loaders
train_loader_50 = DataLoader(
    train_dataset_50,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader_10 = DataLoader(
    val_dataset_10,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Training patients:", len(train_dataset_50))
print("Validation patients:", len(val_dataset_10))


Training patients: 50
Validation patients: 10


In [ ]:
# num_epochs = 1

# unet.train()
# train_loss = 0.0

# for image, target in train_loader_50:
#     image = image.to(device)
#     target = target.to(device)

#     optimizer.zero_grad()

#     output = unet(image)
#     target_cropped = crop_target(target, output.shape)

#     loss = dice_loss(output, target_cropped)
#     loss.backward()
#     optimizer.step()

#     train_loss += loss.item()

# avg_train_loss = train_loss / len(train_loader_50)
# print(f"Training Dice Loss (50 patients, 1 epoch): {avg_train_loss:.4f}")


In [ ]:
ce_loss_fn = torch.nn.CrossEntropyLoss()

def combined_loss(pred, target, dice_weight=0.5):
    """
    pred: (B, C, D, H, W) logits
    target: (B, D, H, W) labels
    """
    # Cross entropy expects (B, C, ...)
    ce = ce_loss_fn(pred, target)

    # Dice loss (already defined earlier)
    dice = dice_loss(pred, target)

    return dice_weight * dice + (1 - dice_weight) * ce


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    unet.train()
    epoch_loss = 0.0

    for image, target in train_loader_50:
        image = image.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        output = unet(image)
        target_cropped = crop_target(target, output.shape)

        loss = combined_loss(output, target_cropped)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader_50)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Combined Loss: {avg_loss:.4f}")


Epoch [1/10] - Combined Loss: 1.1048
Epoch [2/10] - Combined Loss: 0.7027
Epoch [3/10] - Combined Loss: 0.6291
Epoch [4/10] - Combined Loss: 0.5568
Epoch [5/10] - Combined Loss: 0.5322
Epoch [6/10] - Combined Loss: 0.5105
Epoch [7/10] - Combined Loss: 0.4758
Epoch [8/10] - Combined Loss: 0.4439
Epoch [9/10] - Combined Loss: 0.4142
Epoch [10/10] - Combined Loss: 0.3814


In [ ]:
import numpy as np
from scipy.spatial.distance import cdist
import numpy as np
from scipy.spatial.distance import cdist
def hd95(pred, target, voxel_spacing=(1.0, 1.0, 1.0)):
    """
    pred, target: binary masks (D, H, W)
    """
    pred_points = np.argwhere(pred)
    target_points = np.argwhere(target)

    if len(pred_points) == 0 or len(target_points) == 0:
        return np.inf  # worst case

    # Convert voxel indices to physical space
    pred_points = pred_points * np.array(voxel_spacing)
    target_points = target_points * np.array(voxel_spacing)

    distances = cdist(pred_points, target_points)

    hd_pred = np.percentile(distances.min(axis=1), 95)
    hd_target = np.percentile(distances.min(axis=0), 95)

    return max(hd_pred, hd_target)


In [ ]:
unet.eval()

image, target = next(iter(val_loader_10))
image = image.to(device)
target = target.to(device)

with torch.no_grad():
    output = unet(image)

# Crop target to match output
target_cropped = crop_target(target, output.shape)

# Convert to numpy
pred = torch.argmax(output, dim=1).cpu().numpy()[0]
gt = target_cropped.cpu().numpy()[0]

print("Evaluation on 1 validation patient")


Evaluation on 1 validation patient


In [ ]:
# Whole Tumor = classes 1, 2, 3
pred_wt = np.isin(pred, [1, 2, 3])
gt_wt = np.isin(gt, [1, 2, 3])

dice_wt = dice_score(pred, gt, class_id=1)  # rough proxy
hd95_wt = hd95(pred_wt, gt_wt)

print(f"Dice (Whole Tumor): {dice_wt:.4f}")
print(f"HD95 (Whole Tumor): {hd95_wt:.2f} mm")

TypeError: dice_loss() got an unexpected keyword argument 'class_id'

In [ ]:
def dice_score(pred, target, class_id, smooth=1e-5):
    """
    pred: (D, H, W) predicted labels
    target: (D, H, W) ground truth labels
    class_id: integer representing the class to calculate Dice for
    """
    pred_c = (pred == class_id).astype(float)
    target_c = (target == class_id).astype(float)

    intersection = (pred_c * target_c).sum()
    union = pred_c.sum() + target_c.sum()

    return (2. * intersection + smooth) / (union + smooth)

In [ ]:
dice_scores = []
hd95_scores = []

for i, (image, target) in enumerate(val_loader_10):
    if i == 3:
        break

    image = image.to(device)
    target = target.to(device)

    with torch.no_grad():
        output = unet(image)

    target_cropped = crop_target(target, output.shape)

    pred = torch.argmax(output, dim=1).cpu().numpy()[0]
    gt = target_cropped.cpu().numpy()[0]

    pred_wt = np.isin(pred, [1, 2, 3])
    gt_wt = np.isin(gt, [1, 2, 3])

    dice_scores.append(dice_score(pred, gt, class_id=1))
    hd95_scores.append(hd95(pred_wt, gt_wt))

print("Average Dice (WT):", np.mean(dice_scores))
print("Average HD95 (WT):", np.mean(hd95_scores))


In [ ]:
import torch.nn as nn

class UNet3D_MC(nn.Module):
    def __init__(self, in_channels, num_classes, dropout_p=0.3):
        super().__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv3d(in_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Dropout3d(dropout_p),
                nn.Conv3d(out_c, out_c, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Dropout3d(dropout_p),
            )

        self.enc1 = conv_block(in_channels, 16)
        self.pool1 = nn.MaxPool3d(2)

        self.enc2 = conv_block(16, 32)
        self.pool2 = nn.MaxPool3d(2)

        self.bottleneck = conv_block(32, 64)

        self.up2 = nn.ConvTranspose3d(64, 32, 2, stride=2)
        self.dec2 = conv_block(64, 32)

        self.up1 = nn.ConvTranspose3d(32, 16, 2, stride=2)
        self.dec1 = conv_block(32, 16)

        self.out_conv = nn.Conv3d(16, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        d2 = self.up2(b)
        e2 = center_crop(e2, d2.shape[2:])
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        e1 = center_crop(e1, d1.shape[2:])
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out_conv(d1)


In [ ]:
mc_unet = UNet3D_MC(in_channels=4, num_classes=4, dropout_p=0.3)
mc_unet.load_state_dict(unet.state_dict())  # reuse trained weights
mc_unet = mc_unet.to(device)

print("MC Dropout model ready")


NameError: name 'UNet3D_MC' is not defined